## 1. Random Label Sanity Check

In [1]:
%matplotlib inline

import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score

from src.utils.paths import load_paths
from src.utils.logging import setup_logger
from src.pipeline.data_preparation import load_and_prepare_data
from src.pipeline.feature_pipeline import FeaturePipeline

paths = load_paths()
logger = setup_logger(level="INFO")

# Load data
df_all = load_and_prepare_data()

df_train = df_all[df_all["split"] == "train"].copy()
df_val = df_all[df_all["split"] == "val"].copy()

2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Loading VNAT (PCAP-based)...
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | [VNAT] Loaded features.parquet (33711 flows, splits already assigned)
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Loading ISCX (PCAP-based)...
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | [ISCX] Loaded features.parquet (76687 flows, splits already assigned)
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Loading USBVPN (JSON-based)...
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Removing exact duplicate flows across feature columns...
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Ensuring numeric dtypes for feature columns...
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | ✓ All feature columns successfully converted to numeric dtypes
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Removed 5750 duplicate flows (7.92%)
2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Metadata columns present for analysis only: ['source_capture_id', 'source_file']
2026-03-30 12:50

In [2]:
# --- Randomize Labels ---

logger.info("Shuffling training labels...")

y_train_shuffled = df_train["label"].sample(
    frac=1,
    random_state=42
).values

2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Shuffling training labels...


In [3]:
# Fit pipeline and transform

pipeline = FeaturePipeline().fit(df_train)

X_train = pipeline.transform(df_train)
X_val = pipeline.transform(df_val)

feature_cols = pipeline.model_feature_names()

In [4]:
# Train model

logger.info("Training model on shuffled labels...")

model = lgb.LGBMClassifier(random_state=42)

model.fit(
    X_train[feature_cols],
    y_train_shuffled
)

2026-03-30 12:50:17 | INFO | ai-vpn-firewall | Training model on shuffled labels...
[LightGBM] [Info] Number of positive: 10349, number of negative: 39157
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.001331 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 1785
[LightGBM] [Info] Number of data points in the train set: 49506, number of used features: 7
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.209045 -> initscore=-1.330689
[LightGBM] [Info] Start training from score -1.330689


,boosting_type,'gbdt'
,num_leaves,31
,max_depth,-1
,learning_rate,0.1
,n_estimators,100
,subsample_for_bin,200000
,objective,None
,class_weight,None
,min_split_gain,0.0
,min_child_weight,0.001
,min_child_samples,20


In [5]:
# Evaluate

p_val = model.predict_proba(
    X_val[feature_cols]
)[:, 1]

y_val = df_val["label"].values

auc = roc_auc_score(
    y_val,
    p_val
)

logger.info(f"AUC on validation set with random labels: {auc:.4f}")

2026-03-30 12:50:20 | INFO | ai-vpn-firewall | AUC on validation set with random labels: 0.5492


In [6]:
if abs(auc - 0.5) < 0.05:
    logger.info("SUCCESS: AUC is close to 0.5, as expected.")
else:
    logger.error("FAIL: AUC is not close to 0.5. There might be data leakage.")

2026-03-30 12:50:20 | INFO | ai-vpn-firewall | SUCCESS: AUC is close to 0.5, as expected.


## 2. LOOD Evaluation Runner

In [7]:
import sys

sys.path.append(str(paths.repo_root))

from src.cli.run_lood_training import main as run_lood

logger.info("Running LOOD evaluation...")

run_lood()

2026-03-30 12:50:20 | INFO | ai-vpn-firewall | Running LOOD evaluation...
2026-03-30 12:50:20 | INFO | ai-vpn-firewall | Output directory: C:\Users\scoti\PycharmProjects\ai-vpn-firewall\artifacts\lood_training
2026-03-30 12:50:20 | INFO | ai-vpn-firewall | Loading combined dataset...
2026-03-30 12:50:20 | INFO | ai-vpn-firewall | Loading VNAT (PCAP-based)...
2026-03-30 12:50:20 | INFO | ai-vpn-firewall | [VNAT] Loaded features.parquet (33711 flows, splits already assigned)
2026-03-30 12:50:20 | INFO | ai-vpn-firewall | Loading ISCX (PCAP-based)...
2026-03-30 12:50:20 | INFO | ai-vpn-firewall | [ISCX] Loaded features.parquet (76687 flows, splits already assigned)
2026-03-30 12:50:20 | INFO | ai-vpn-firewall | Loading USBVPN (JSON-based)...
2026-03-30 12:50:21 | INFO | ai-vpn-firewall | Removing exact duplicate flows across feature columns...
2026-03-30 12:50:21 | INFO | ai-vpn-firewall | Ensuring numeric dtypes for feature columns...
2026-03-30 12:50:21 | INFO | ai-vpn-firewall | ✓ All 

0